# DroneTask Design-Check Notebook

## Randomization and counterbalancing audit

This notebook distinguishes structural design features that apply to all participants from participant-level random realizations generated at runtime. The task currently uses ordinary random shuffling for nuisance randomization, exact counterbalancing for distance-2 slider probes, and a fixed block-condition order unless/until Latin Square block-order counterbalancing is implemented.

| Mechanism | Runtime logic in latest scripts | File / location | Randomization type | Targeted design-check output |
|---|---|---|---|---|
| PNG stimulus assignment | jsPsych.randomization.shuffle(STIMULUS_IMAGE_POOL.slice()); the 200 PNGs are shuffled once per participant and assigned sequentially to 4 blocks × 50 trials | index.html, using STIMULUS_IMAGE_POOL from stimuli.js | Random permutation without replacement | stimulus_ribbons.html; image_assignment_to_serial_positions.html; memory_probe_image_assignment.html |
| Main block order | Current active logic uses fixed condition arrays: factors_vol = [49, 4, 49, 4], factors_stc = [16, 64, 16, 64], factors_valence = ['reward', 'reward', 'loss', 'loss'] | stimuli.js, consumed by index.html | Fixed order; Latin Square not yet implemented | block_condition_design.html; change_point_map.html; trajectory audit HTMLs |
| Reward/loss shifted trajectory assignment | reward_uses_shifted_sequences = Math.random() < 0.5; either reward blocks use shifted high/low trajectories and loss blocks use base trajectories, or vice versa | stimuli.js | Two-branch Bernoulli randomization at participant/page-load level | trajectories_reward_shifted_true.html; trajectories_reward_shifted_false.html |
| Memory-pair order | jsPsych.randomization.shuffle(PREDEFINED_PAIRS.slice()) separately within each block | memory_task.js | Random permutation without replacement | pair_order_pseudorandomization.html |
| Earlier/later slider-probe assignment | For the 8 distance-2 pairs, shuffle the pair list, then assign earlier/later by alternating index parity; distance-1 pairs use the only intervening item | memory_task.js | Randomized exact within-block counterbalancing | slider_probe_counterbalance_summary.html; slider_probe_counterbalance_exact_within_block.html; slider_probe_pair_level_assignment_probability.html |
| Temporal-order left/right display | jsPsych.randomization.shuffle([stim1, stim2]); response mapping remains fixed as 1 = left, 2 = right | memory_task.js | Random two-item permutation | Covered in expected CSV/data-dictionary checks; optional future output: temporal_order_left_right_balance.html |

## Latin Square scope.  
Latin Square counterbalancing is appropriate for the four main block conditions, for the purpose of block-order counterbalancing. It should not replace PNG shuffling, memory-pair order shuffling, earlier/later probe balancing, or temporal-order left/right display randomization.

In [38]:
import re, json, hashlib, os, math, warnings, textwrap
from pathlib import Path
from datetime import datetime
from collections import Counter, OrderedDict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

REPO_ROOT = Path(".")
DESIGN_DIR = REPO_ROOT / "design_checks"
DESIGN_DIR.mkdir(exist_ok=True)

# ── Matplotlib house style ──────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})

# ── JS parsers ──────────────────────────────────────────────────

def read_text(relpath):
    p = REPO_ROOT / relpath
    return p.read_text(encoding="utf-8") if p.exists() else None

def parse_js_string(text, varname):
    m = re.search(rf'var\s+{varname}\s*=\s*"([^"]*)"', text)
    if m: return m.group(1)
    m = re.search(rf"var\s+{varname}\s*=\s*'([^']*)'", text)
    return m.group(1) if m else None

def parse_js_flat_array(text, varname):
    pat = rf'var\s+{varname}\s*=\s*\[([\s\S]*?)\];'
    m = re.search(pat, text)
    if not m: return None
    return [float(x) for x in re.findall(r'-?\d+\.?\d*', m.group(1))]

def parse_js_2d_array(text, varname):
    pat = rf'var\s+{varname}\s*=\s*\[([\s\S]*?)\];'
    m = re.search(pat, text)
    if not m: return None
    rows = re.findall(r'\[([^\]]+)\]', m.group(1))
    return [[float(x) for x in re.findall(r'-?\d+\.?\d*', row)] for row in rows]

def parse_js_string_array(text, varname):
    pat = rf'var\s+{varname}\s*=\s*\[([\s\S]*?)\];'
    m = re.search(pat, text)
    if not m: return None
    return re.findall(r'"([^"]*)"', m.group(1))

def parse_js_int_pair_array(text, varname):
    pat = rf'var\s+{varname}\s*=\s*\[([\s\S]*?)\];'
    m = re.search(pat, text)
    if not m: return None
    pairs = re.findall(r'\[\s*(\d+)\s*,\s*(\d+)\s*\]', m.group(1))
    return [[int(a), int(b)] for a, b in pairs]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def save_fig(fig, stem, also_html=False):
    png = DESIGN_DIR / f"{stem}.png"
    fig.savefig(png, bbox_inches="tight")
    print(f"  \u2713 Exported {png}")
    if also_html:
        # wrap the PNG in a minimal HTML
        import base64
        b64 = base64.b64encode(png.read_bytes()).decode()
        html_path = DESIGN_DIR / f"{stem}.html"
        html_path.write_text(
            f'<!DOCTYPE html><html><body style="margin:0;background:#fff">'
            f'<img src="data:image/png;base64,{b64}" style="max-width:100%"></body></html>',
            encoding="utf-8")
        print(f"  \u2713 Exported {html_path}")

def save_html(html_str, filename):
    path = DESIGN_DIR / filename
    path.write_text(html_str, encoding="utf-8")
    print(f"  \u2713 Exported {path}")

def pair_in(pair, plist):
    return any(pair[0] == p[0] and pair[1] == p[1] for p in plist)

def styled_table_css(uid="tbl"):
    return textwrap.dedent(f"""\
    <style>
    #{uid} th {{ background:#f3f3f3; color:#222; font-weight:650;
      border:1px solid #d6d6d6; padding:5px 10px; text-align:left; }}
    #{uid} td {{ border:1px solid #e1e1e1; padding:4px 10px;
      font-variant-numeric:tabular-nums; }}
    #{uid} tr:nth-child(even) td {{ background:#fafafa; }}
    #{uid} table {{ border-collapse:collapse; font-size:12px;
      font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif; }}
    #{uid} caption {{ caption-side:top; padding:4px 0; color:#444;
      font-size:12px; text-align:left; font-weight:600; }}
    </style>
    """)

print("Helpers loaded.")

Helpers loaded.


## A. Source-file and version audit

In [39]:
FILES_TO_CHECK = [
    "static/task/stimuli.js",
    "static/task/memory_task.js",
    "static/task/stimuli-details.js",
    "static/task/trial.js",
    "index.html",
]
rows = []
for fp in FILES_TO_CHECK:
    p = Path(fp)
    exists = p.exists()
    mtime = datetime.fromtimestamp(p.stat().st_mtime).isoformat() if exists else None
    h = sha256(p)[:16] + "..." if exists else None
    rows.append(dict(file=fp, exists=exists, modified=mtime, sha256_prefix=h))

file_audit = pd.DataFrame(rows)
display(file_audit)

stimuli_src = read_text("static/task/stimuli.js")
memory_src  = read_text("static/task/memory_task.js")
details_src = read_text("static/task/stimuli-details.js")
trial_src   = read_text("static/task/trial.js")
index_src   = read_text("index.html")

stim_version = parse_js_string(stimuli_src, "stimuli_version")
print(f"\nstimuli_version = {stim_version!r}")
assert "png" in (stim_version or "").lower(), "Expected PNG-based stimuli version"
print("\u2713 Confirmed PNG-based stimulus design.")

,file,exists,modified,sha256_prefix
0,static/task/stimuli.js,True,2026-05-14T02:27:56.978934,f343e15625f4f466...
1,static/task/memory_task.js,True,2026-06-15T08:40:17.626902,52f6bfd05851f79f...
2,static/task/stimuli-details.js,True,2026-04-22T01:37:23.717670,58ce22e3de0b09d5...
3,static/task/trial.js,True,2026-05-14T02:27:56.979280,0fed9d0555fa244a...
4,index.html,True,2026-06-15T16:49:17.506676,b7cddd84cfb797da...



stimuli_version = 'v10-png-main-practice-emoji'
✓ Confirmed PNG-based stimulus design.


## C. Block condition design

In [4]:
factors_vol = parse_js_flat_array(stimuli_src, "factors_vol")
factors_stc = parse_js_flat_array(stimuli_src, "factors_stc")
fv_m = re.search(r"var\s+factors_valence\s*=\s*\[(.*?)\]", stimuli_src)
factors_valence = re.findall(r"'(\w+)'", fv_m.group(1)) if fv_m else []

block_df = pd.DataFrame({
    "block": [1,2,3,4],
    "valence": factors_valence,
    "vol_param": [int(v) for v in factors_vol],
    "vol_level": ["high" if v==49 else "low" for v in factors_vol],
    "stc_param": [int(s) for s in factors_stc],
    "stc_level": ["high" if s==64 else "low" for s in factors_stc],
})
display(block_df)

for blk, val, vol, stc in [(1,"reward",49,16),(2,"reward",4,64),(3,"loss",49,16),(4,"loss",4,64)]:
    r = block_df[block_df.block==blk].iloc[0]
    assert r.valence==val and r.vol_param==vol and r.stc_param==stc
print("\u2713 Block condition design matches specification.")

,block,valence,vol_param,vol_level,stc_param,stc_level
0,1,reward,49,high,16,low
1,2,reward,4,low,64,high
2,3,loss,49,high,16,low
3,4,loss,4,low,64,high


✓ Block condition design matches specification.


## Latin-square block-order counterbalancing

This section audits the implemented block-order counterbalancing logic directly from `index.html`.

The canonical task conditions remain defined in `stimuli.js`:

- canonical index `0`: reward, high volatility / low stochasticity

- canonical index `1`: reward, low volatility / high stochasticity

- canonical index `2`: loss, high volatility / low stochasticity

- canonical index `3`: loss, low volatility / high stochasticity

The Latin Square in `index.html` does not change the latent position sequences themselves. Instead, it changes which canonical condition is presented as displayed Block 1, Block 2, Block 3, and Block 4. Therefore:

- `display_block` = ordinal block experienced by the participant

- `design_idx` / `canonical_design_idx` = canonical condition identity

- `condition_id` = human-readable canonical condition label

- `latin_square_group` = assigned Latin-square row

- `latin_square_order` = full participant-level condition order

This audit verifies that each Latin-square row contains all four canonical conditions exactly once, and that each canonical condition appears exactly once in each displayed block position across the four rows.

In [40]:
# ============================================================
# Latin-square extraction from implemented index.html
# ============================================================

import re
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Use existing parsed source variables if they already exist in the notebook.
# Otherwise read directly from repo-local files.
try:
    index_src
except NameError:
    index_path = Path("index.html")
    assert index_path.exists(), "index.html not found. Run this notebook from the repo root or set index_path manually."
    index_src = index_path.read_text(encoding="utf-8")

try:
    stimuli_src
except NameError:
    stimuli_path = Path("static/task/stimuli.js")
    assert stimuli_path.exists(), "static/task/stimuli.js not found. Run this notebook from the repo root or set stimuli_path manually."
    stimuli_src = stimuli_path.read_text(encoding="utf-8")


def extract_js_array_literal(src, var_name):
    """
    Extract a simple JavaScript array literal assigned as:
      var VAR_NAME = [...];

    Works for the current numeric/string/nested-array design constants.
    """
    pattern = r"var\s+" + re.escape(var_name) + r"\s*=\s*(\[[\s\S]*?\]);"
    m = re.search(pattern, src)

    if not m:
        raise ValueError(f"Could not find JS array variable: {var_name}")

    literal = m.group(1)

    # Current arrays are valid Python literals after normalizing JS strings.
    try:
        return ast.literal_eval(literal)
    except Exception as e:
        raise ValueError(f"Could not parse {var_name}. Extracted literal:\n{literal[:500]}") from e


# Extract implemented Latin-square constants directly from index.html
LATIN_SQUARE_4_EXTRACTED = extract_js_array_literal(index_src, "LATIN_SQUARE_4")
CANONICAL_CONDITION_LABELS_EXTRACTED = extract_js_array_literal(index_src, "CANONICAL_CONDITION_LABELS")

# Extract canonical factor definitions directly from stimuli.js
FACTORS_VOL_EXTRACTED = extract_js_array_literal(stimuli_src, "factors_vol")
FACTORS_STC_EXTRACTED = extract_js_array_literal(stimuli_src, "factors_stc")
FACTORS_VALENCE_EXTRACTED = extract_js_array_literal(stimuli_src, "factors_valence")

print("Extracted LATIN_SQUARE_4 from index.html:")
display(pd.DataFrame(LATIN_SQUARE_4_EXTRACTED, index=[f"group_{i}" for i in range(len(LATIN_SQUARE_4_EXTRACTED))]))

print("Extracted canonical condition labels from index.html:")
display(pd.DataFrame({
    "canonical_design_idx": range(len(CANONICAL_CONDITION_LABELS_EXTRACTED)),
    "condition_id": CANONICAL_CONDITION_LABELS_EXTRACTED,
    "vol": FACTORS_VOL_EXTRACTED,
    "stc": FACTORS_STC_EXTRACTED,
    "valence": FACTORS_VALENCE_EXTRACTED,
}))

Extracted LATIN_SQUARE_4 from index.html:


,0,1,2,3
group_0,0,1,3,2
group_1,1,2,0,3
group_2,2,3,1,0
group_3,3,0,2,1


Extracted canonical condition labels from index.html:


,canonical_design_idx,condition_id,vol,stc,valence
0,0,A_reward_highVol_lowStc,49,16,reward
1,1,B_reward_lowVol_highStc,4,64,reward
2,2,C_loss_highVol_lowStc,49,16,loss
3,3,D_loss_lowVol_highStc,4,64,loss


In [41]:
# ============================================================
# Latin-square invariant checks
# ============================================================

latin = LATIN_SQUARE_4_EXTRACTED
condition_labels = CANONICAL_CONDITION_LABELS_EXTRACTED

n_groups = len(latin)
n_conditions = len(condition_labels)

assert n_groups == 4, f"Expected 4 Latin-square rows, found {n_groups}."
assert n_conditions == 4, f"Expected 4 canonical conditions, found {n_conditions}."

# Each row must be a permutation of 0, 1, 2, 3.
expected_set = set(range(n_conditions))

for group_idx, row in enumerate(latin):
    assert len(row) == n_conditions, f"Latin group {group_idx} has wrong length: {row}"
    assert set(row) == expected_set, f"Latin group {group_idx} is not a permutation of 0..3: {row}"

# Each displayed block position must contain each canonical condition exactly once across groups.
latin_arr = np.array(latin)

for display_pos in range(n_conditions):
    col = latin_arr[:, display_pos]
    assert set(col) == expected_set, (
        f"Displayed block position {display_pos + 1} does not contain all conditions exactly once: {col}"
    )

print("✓ Each Latin-square row contains all four canonical conditions exactly once.")
print("✓ Each displayed block position receives each canonical condition exactly once across groups.")

# Build long reviewer-facing audit table
latin_rows = []

for group_idx, row in enumerate(latin):
    for display_pos, design_idx in enumerate(row, start=1):
        latin_rows.append(dict(
            latin_square_group=group_idx,
            displayed_block=display_pos,
            canonical_design_idx=design_idx,
            condition_id=condition_labels[design_idx],
            valence=FACTORS_VALENCE_EXTRACTED[design_idx],
            volatility="high" if FACTORS_VOL_EXTRACTED[design_idx] == 49 else "low",
            stochasticity="high" if FACTORS_STC_EXTRACTED[design_idx] == 64 else "low",
            vol_param=FACTORS_VOL_EXTRACTED[design_idx],
            stc_param=FACTORS_STC_EXTRACTED[design_idx],
        ))

latin_df = pd.DataFrame(latin_rows)

display(latin_df)

# Compact matrix view: rows = Latin group, columns = displayed block position
latin_matrix_display = (
    latin_df
    .assign(cell=lambda d: d["condition_id"].str.replace("_", " ", regex=False))
    .pivot(index="latin_square_group", columns="displayed_block", values="cell")
)

display(latin_matrix_display)

# Position-balance check table
position_balance = (
    latin_df
    .groupby(["displayed_block", "condition_id"])
    .size()
    .reset_index(name="count")
)

display(position_balance)

assert (position_balance["count"] == 1).all()
print("✓ Position-balance table confirms count = 1 for every displayed block × condition cell.")

✓ Each Latin-square row contains all four canonical conditions exactly once.
✓ Each displayed block position receives each canonical condition exactly once across groups.


,latin_square_group,displayed_block,canonical_design_idx,condition_id,valence,volatility,stochasticity,vol_param,stc_param
0,0,1,0,A_reward_highVol_lowStc,reward,high,low,49,16
1,0,2,1,B_reward_lowVol_highStc,reward,low,high,4,64
2,0,3,3,D_loss_lowVol_highStc,loss,low,high,4,64
3,0,4,2,C_loss_highVol_lowStc,loss,high,low,49,16
4,1,1,1,B_reward_lowVol_highStc,reward,low,high,4,64
5,1,2,2,C_loss_highVol_lowStc,loss,high,low,49,16
6,1,3,0,A_reward_highVol_lowStc,reward,high,low,49,16
7,1,4,3,D_loss_lowVol_highStc,loss,low,high,4,64
8,2,1,2,C_loss_highVol_lowStc,loss,high,low,49,16
9,2,2,3,D_loss_lowVol_highStc,loss,low,high,4,64


displayed_block,1,2,3,4
latin_square_group,,,,
0,A reward highVol lowStc,B reward lowVol highStc,D loss lowVol highStc,C loss highVol lowStc
1,B reward lowVol highStc,C loss highVol lowStc,A reward highVol lowStc,D loss lowVol highStc
2,C loss highVol lowStc,D loss lowVol highStc,B reward lowVol highStc,A reward highVol lowStc
3,D loss lowVol highStc,A reward highVol lowStc,C loss highVol lowStc,B reward lowVol highStc


,displayed_block,condition_id,count
0,1,A_reward_highVol_lowStc,1
1,1,B_reward_lowVol_highStc,1
2,1,C_loss_highVol_lowStc,1
3,1,D_loss_lowVol_highStc,1
4,2,A_reward_highVol_lowStc,1
5,2,B_reward_lowVol_highStc,1
6,2,C_loss_highVol_lowStc,1
7,2,D_loss_lowVol_highStc,1
8,3,A_reward_highVol_lowStc,1
9,3,B_reward_lowVol_highStc,1


✓ Position-balance table confirms count = 1 for every displayed block × condition cell.


In [42]:
# ============================================================
# Latin-square visualization
# ============================================================

# Short labels for plot readability
short_label_map = {
    "A_reward_highVol_lowStc": "A\nReward\nHigh vol / Low stc",
    "B_reward_lowVol_highStc": "B\nReward\nLow vol / High stc",
    "C_loss_highVol_lowStc": "C\nLoss\nHigh vol / Low stc",
    "D_loss_lowVol_highStc": "D\nLoss\nLow vol / High stc",
}

latin_df["plot_label"] = latin_df["condition_id"].map(short_label_map).fillna(latin_df["condition_id"])

plot_grid = (
    latin_df
    .pivot(index="latin_square_group", columns="displayed_block", values="plot_label")
    .sort_index()
)

idx_grid = (
    latin_df
    .pivot(index="latin_square_group", columns="displayed_block", values="canonical_design_idx")
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10.8, 5.6))

# Numeric grid used only to create a stable visual tile layout.
im = ax.imshow(idx_grid.values, aspect="auto", cmap="Pastel1", vmin=0, vmax=3)

ax.set_xticks(np.arange(4))
ax.set_xticklabels([f"Displayed block {i}" for i in range(1, 5)], fontsize=10)
ax.set_yticks(np.arange(4))
ax.set_yticklabels([f"Latin group {i}" for i in range(4)], fontsize=10)

# Cell text
for r in range(plot_grid.shape[0]):
    for c in range(plot_grid.shape[1]):
        ax.text(
            c,
            r,
            plot_grid.iloc[r, c],
            ha="center",
            va="center",
            fontsize=9.5,
            color="#1f2937",
            fontweight="bold",
            linespacing=1.15,
        )

# Grid lines
ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=2.2)
ax.tick_params(which="minor", bottom=False, left=False)

ax.set_title(
    "Latin-square counterbalancing of block order",
    fontsize=15,
    fontweight="bold",
    pad=14,
)

ax.text(
    -0.48,
    4.08,
    "Rows are participant groups; columns are experienced block positions. "
    "Each condition appears once per row and once per displayed block position.",
    fontsize=10,
    color="#475569",
    ha="left",
    va="top",
)

ax.set_xlabel("Ordinal block position experienced by participant", fontsize=11)
ax.set_ylabel("Counterbalancing group", fontsize=11)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.tight_layout()

# Display inside notebook
plt.show()

# Export with existing notebook helper if available
try:
    save_fig(fig, "latin_square_block_order", also_html=True)
except NameError:
    fig.savefig("latin_square_block_order.png", dpi=220, bbox_inches="tight")
    print("save_fig not found; saved latin_square_block_order.png instead.")

# Reviewer-facing HTML table
latin_html = (
    '<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Arial,sans-serif;'
    'padding:18px;color:#222;background:#fff;">'
    '<h2 style="margin:0 0 8px 0;">Latin-square block-order counterbalancing</h2>'
    '<p style="max-width:960px;line-height:1.45;color:#475569;font-size:13px;">'
    'This table is extracted from the implemented <code>index.html</code> Latin-square constants. '
    'Rows are participant groups; columns are displayed block positions. '
    'The cell value is the canonical task condition presented in that block position.'
    '</p>'
    + latin_matrix_display.to_html()
    + '<h3>Long-form audit table</h3>'
    + latin_df[[
        "latin_square_group",
        "displayed_block",
        "canonical_design_idx",
        "condition_id",
        "valence",
        "volatility",
        "stochasticity",
        "vol_param",
        "stc_param",
    ]].to_html(index=False)
    + '</div>'
)

try:
    save_html(latin_html, "latin_square_block_order.html")
    print("✓ Exported latin_square_block_order.html")
except NameError:
    Path("latin_square_block_order.html").write_text(latin_html, encoding="utf-8")
    print("save_html not found; wrote latin_square_block_order.html directly.")

  ✓ Exported design_checks/latin_square_block_order.png
  ✓ Exported design_checks/latin_square_block_order.html
  ✓ Exported design_checks/latin_square_block_order.html
✓ Exported latin_square_block_order.html


## D. Reward/loss drone trajectory shifts
Plots show **base** (dotted) vs **assembled** (solid) bird/bag sequences,
with change-point locations as vertical dashed lines.

In [5]:
main_bird = parse_js_2d_array(stimuli_src, "main_bird_position")
main_bag  = parse_js_2d_array(stimuli_src, "main_bag_position")
assert len(main_bird)==4 and len(main_bag)==4
for i in range(4):
    assert len(main_bird[i])==50 and len(main_bag[i])==50
    assert all(np.isfinite(main_bird[i])) and all(np.isfinite(main_bag[i]))
print("\u2713 Trajectory arrays: 4 blocks x 50 trials, all finite.")

HIGHVOL_SHIFT = float(re.search(r"var\s+HIGHVOL_SHIFT_DELTA\s*=\s*([\-\d.]+)", stimuli_src).group(1))
LOWVOL_SHIFT  = float(re.search(r"var\s+LOWVOL_SHIFT_DELTA\s*=\s*([\-\d.]+)", stimuli_src).group(1))

hv_base_bird = np.array(main_bird[0]); lv_base_bird = np.array(main_bird[1])
hv_base_bag  = np.array(main_bag[0]);  lv_base_bag  = np.array(main_bag[1])

def shift_seq(seq, delta): return np.clip(seq + delta, 10, 90)

hv_shifted_bird = shift_seq(hv_base_bird, HIGHVOL_SHIFT)
hv_shifted_bag  = shift_seq(hv_base_bag,  HIGHVOL_SHIFT)
lv_shifted_bird = shift_seq(lv_base_bird, LOWVOL_SHIFT)
lv_shifted_bag  = shift_seq(lv_base_bag,  LOWVOL_SHIFT)

CP_HIGH_VOL = [3, 5, 11, 19, 25, 37, 39, 47]
CP_LOW_VOL  = [3, 5, 11, 15, 19, 21, 25, 29, 31, 37, 39, 45, 47]

def plot_trajectories(reward_shifted, save_stem):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    fig.suptitle(
        f"Latent trajectories  \u00b7  reward_uses_shifted_sequences = {reward_shifted}",
        fontsize=15, fontweight="bold", y=0.98)

    # Build assembled sequences for this branch
    if reward_shifted:
        assembled_bird = [hv_shifted_bird, lv_shifted_bird, hv_base_bird, lv_base_bird]
        assembled_bag  = [hv_shifted_bag,  lv_shifted_bag,  hv_base_bag,  lv_base_bag]
        deltas = [HIGHVOL_SHIFT, LOWVOL_SHIFT, 0, 0]
    else:
        assembled_bird = [hv_base_bird, lv_base_bird, hv_shifted_bird, lv_shifted_bird]
        assembled_bag  = [hv_base_bag,  lv_base_bag,  hv_shifted_bag,  lv_shifted_bag]
        deltas = [0, 0, HIGHVOL_SHIFT, LOWVOL_SHIFT]

    bases_bird = [hv_base_bird, lv_base_bird, hv_base_bird, lv_base_bird]
    bases_bag  = [hv_base_bag,  lv_base_bag,  hv_base_bag,  lv_base_bag]

    trials = np.arange(1, 51)

    for idx in range(4):
        ax = axes[idx // 2][idx % 2]
        vol = int(factors_vol[idx])
        cps = CP_HIGH_VOL if vol == 49 else CP_LOW_VOL
        vlabel = factors_valence[idx]
        vol_str = "high" if vol == 49 else "low"
        stc_str = "high" if int(factors_stc[idx]) == 64 else "low"
        d = deltas[idx]
        shift_label = f"shifted (\u0394=+{int(d)})" if d > 0 else (f"shifted (\u0394={int(d)})" if d < 0 else "base (\u0394=+0)")

        # change-point lines
        for cp in cps:
            ax.axvline(cp, color="#999", ls="--", lw=0.7, alpha=0.55)

        # base (dotted)
        ax.plot(trials, bases_bird[idx], ls=":", lw=1.3, color="#1f77b4", alpha=0.55, label="base bird")
        ax.plot(trials, bases_bag[idx],  ls=":", lw=1.3, color="#ff7f0e", alpha=0.55, label="base bag")
        # assembled (solid)
        ax.plot(trials, assembled_bird[idx], ls="-", lw=1.8, color="#1f77b4", label="assembled bird")
        ax.plot(trials, assembled_bag[idx],  ls="-", lw=1.8, color="#ff7f0e", label="assembled bag")

        ax.set_title(
            f"design idx {idx}  \u00b7  vol={vol_str}, stc={stc_str}, valence={vlabel}\n{shift_label}",
            fontsize=11, color="#c00" if vlabel == "reward" else "#333")
        ax.set_ylim(0, 100)
        ax.set_ylabel("position (%)")
        if idx >= 2: ax.set_xlabel("trial within block")
        if idx == 0: ax.legend(fontsize=8, loc="upper right", framealpha=0.85)

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save_fig(fig, save_stem, also_html=True)
    plt.show()

plot_trajectories(True,  "trajectories_reward_shifted_true")
plot_trajectories(False, "trajectories_reward_shifted_false")
print("\u2713 Trajectory plots exported.")

✓ Trajectory arrays: 4 blocks x 50 trials, all finite.
  ✓ Exported design_checks/trajectories_reward_shifted_true.png
  ✓ Exported design_checks/trajectories_reward_shifted_true.html
  ✓ Exported design_checks/trajectories_reward_shifted_false.png
  ✓ Exported design_checks/trajectories_reward_shifted_false.html
✓ Trajectory plots exported.


## E. Change-point / boundary map

In [6]:
cp_summary_rows = []
for blk in range(1, 5):
    vol = int(factors_vol[blk-1])
    cps = CP_HIGH_VOL if vol == 49 else CP_LOW_VOL
    cp_summary_rows.append(dict(
        block=blk, vol_level="high" if vol==49 else "low",
        n_change_points=len(cps), change_points=str(cps)))
cp_summary = pd.DataFrame(cp_summary_rows)
display(cp_summary)

fig, ax = plt.subplots(figsize=(13, 3.5))
colors_blk = {1:"#1f77b4", 2:"#2ca02c", 3:"#9467bd", 4:"#d62728"}
for blk in range(1, 5):
    vol = int(factors_vol[blk-1])
    cps = CP_HIGH_VOL if vol == 49 else CP_LOW_VOL
    vol_str = "high" if vol==49 else "low"
    ax.scatter(cps, [blk]*len(cps), s=80, color=colors_blk[blk], zorder=5,
               label=f"Block {blk} ({vol_str} vol)")
ax.set_yticks([1,2,3,4]); ax.set_ylabel("Block"); ax.set_xlabel("Trial (1-indexed)")
ax.set_xlim(0, 51); ax.set_ylim(0.5, 4.5); ax.invert_yaxis()
ax.set_title("Change points by block and trial", fontweight="bold")
ax.legend(fontsize=9, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22))
save_fig(fig, "change_point_map", also_html=True)
plt.show()

,block,vol_level,n_change_points,change_points
0,1,high,8,"[3, 5, 11, 19, 25, 37, 39, 47]"
1,2,low,13,"[3, 5, 11, 15, 19, 21, 25, 29, 31, 37, 39, 45,..."
2,3,high,8,"[3, 5, 11, 19, 25, 37, 39, 47]"
3,4,low,13,"[3, 5, 11, 15, 19, 21, 25, 29, 31, 37, 39, 45,..."


  ✓ Exported design_checks/change_point_map.png
  ✓ Exported design_checks/change_point_map.html


## F. Memory-pair structure

In [25]:
PREDEFINED_PAIRS = parse_js_int_pair_array(memory_src, "PREDEFINED_PAIRS")
SLIDER_PAIRS     = parse_js_int_pair_array(stimuli_src, "SLIDER_PAIRS")
BOUNDARY_MIDDLE  = parse_js_int_pair_array(stimuli_src, "BOUNDARY_MIDDLE_PAIRS")
NONBOUNDARY_MID  = parse_js_int_pair_array(stimuli_src, "NONBOUNDARY_MIDDLE_PAIRS")

print(f"PREDEFINED_PAIRS ({len(PREDEFINED_PAIRS)}): {PREDEFINED_PAIRS}")

def probe_candidate_indices(pair):
    """Return all serial positions between the two endpoint trials."""
    return list(range(pair[0] + 1, pair[1]))

def probe_distance_label(n_candidates):
    if n_candidates == 1:
        return "1 interveneing object"
    if n_candidates == 2:
        return "2 intervening objects"
    return f"d{n_candidates} intervening objects"

def placement_design_label(pair):
    candidates = probe_candidate_indices(pair)
    n = len(candidates)

    if n == 1:
        return "middle probe"
    if n == 2:
        return "earlier or later probe (counterbalanced)"
    return "earlier or later probe (counterbalanced)"

pair_rows = []

for pid, pair in enumerate(PREDEFINED_PAIRS):
    candidates = probe_candidate_indices(pair)
    n_candidates = len(candidates)

    is_slider = pair_in(pair, SLIDER_PAIRS)
    is_bnd    = pair_in(pair, BOUNDARY_MIDDLE)
    is_nbnd   = pair_in(pair, NONBOUNDARY_MID)

    pair_rows.append(dict(
        pair_id=f"P{pid+1:02d}",
        pair=f"({pair[0]}, {pair[1]})",
        endpoint_1=pair[0],
        endpoint_2=pair[1],
        true_distance=n_candidates,
        candidate_indices=str(candidates),
        n_candidates=n_candidates,

        # New design labels
        design_role=placement_design_label(pair),
        probe_distance_type=probe_distance_label(n_candidates),

        # Keep these booleans for possible downstream filtering,
        # but do not display them in the main reviewer table.
        legacy_slider_pair=is_slider,
        legacy_boundary_middle=is_bnd,
        legacy_nonboundary_middle=is_nbnd,
    ))

pair_df = pd.DataFrame(pair_rows)

display_cols = [
    "pair_id",
    "pair",
    "true_distance",
    "n_candidates",
    "candidate_indices",
    "design_role",
    "probe_distance_type",
]

display(pair_df[display_cols])

assert len(PREDEFINED_PAIRS) == 14
d1 = pair_df[pair_df.n_candidates == 1]
d2 = pair_df[pair_df.n_candidates == 2]
assert len(d1) == 6 and len(d2) == 8
assert (pair_df["design_role"].isin([
    "middle probe",
    "earlier or later probe (counterbalanced)",
    "earlier or later probe (counterbalanced)",
])).all()

PREDEFINED_PAIRS (14): [[2, 4], [6, 9], [7, 10], [11, 13], [16, 18], [17, 20], [22, 24], [23, 26], [28, 31], [30, 33], [34, 36], [35, 38], [39, 41], [42, 45]]


,pair_id,pair,true_distance,n_candidates,candidate_indices,design_role,probe_distance_type
0,P01,"(2, 4)",1,1,[3],middle probe,1 interveneing object
1,P02,"(6, 9)",2,2,"[7, 8]",earlier or later probe (counterbalanced),2 intervening objects
2,P03,"(7, 10)",2,2,"[8, 9]",earlier or later probe (counterbalanced),2 intervening objects
3,P04,"(11, 13)",1,1,[12],middle probe,1 interveneing object
4,P05,"(16, 18)",1,1,[17],middle probe,1 interveneing object
5,P06,"(17, 20)",2,2,"[18, 19]",earlier or later probe (counterbalanced),2 intervening objects
6,P07,"(22, 24)",1,1,[23],middle probe,1 interveneing object
7,P08,"(23, 26)",2,2,"[24, 25]",earlier or later probe (counterbalanced),2 intervening objects
8,P09,"(28, 31)",2,2,"[29, 30]",earlier or later probe (counterbalanced),2 intervening objects
9,P10,"(30, 33)",2,2,"[31, 32]",earlier or later probe (counterbalanced),2 intervening objects


### Memory-pair coverage along the trial axis
Horizontal bars show each pair's span; **\u00d7** marks the slider probe item(s).
Vertical lines indicate high-vol change points.

In [ ]:
# -----------------------------
# Configurable setting
# -----------------------------
EXAMPLE_SEED = 20260513
EXAMPLE_BLOCK = 1
rng = np.random.default_rng(EXAMPLE_SEED)

# Use high-vol CP for block 1/3 and low-vol CP for block 2/4.
# If CP_HIGH_VOL / CP_LOW_VOL are already defined earlier, this reuses them.
block_is_high_vol = EXAMPLE_BLOCK in [1, 3]
cp_list = CP_HIGH_VOL if block_is_high_vol else CP_LOW_VOL
cp_label = "high-vol change point" if block_is_high_vol else "low-vol change point"

# -----------------------------
# Match memory_task.js logic
# -----------------------------
def pair_key(pair):
    return f"{pair[0]}_{pair[1]}"

def get_intervening_indices(pair):
    return list(range(pair[0] + 1, pair[1]))

def get_distance2_pairs(pairs):
    return [p for p in pairs if len(get_intervening_indices(p)) == 2]

def init_placement_probe_assignments_for_block(pairs, rng):
    """
    Python mirror of memory_task.js:
    - Take all distance-2 pairs.
    - Shuffle them.
    - Alternate earlier/later probe assignment.
    """
    assignments = {}
    distance2_pairs = get_distance2_pairs(pairs)
    shuffled_idx = rng.permutation(len(distance2_pairs))
    shuffled_pairs = [distance2_pairs[i] for i in shuffled_idx]

    for i, pair in enumerate(shuffled_pairs):
        candidates = get_intervening_indices(pair)
        choose_earlier = (i % 2 == 0)
        assignments[pair_key(pair)] = candidates[0] if choose_earlier else candidates[1]

    return assignments

def choose_placement_probe_index(pair, assignments):
    candidates = get_intervening_indices(pair)

    if len(candidates) == 0:
        return None

    # One-intervening-item pairs are deterministic.
    if len(candidates) == 1:
        return candidates[0]

    # Two-intervening-item pairs use the balanced block-level assignment.
    if len(candidates) == 2:
        return assignments[pair_key(pair)]

    # Fallback only; current design never reaches this.
    return candidates[0]

# Pair order is also pseudo-randomized within block.
pair_order_idx = rng.permutation(len(PREDEFINED_PAIRS))
ordered_pairs = [PREDEFINED_PAIRS[i] for i in pair_order_idx]

# Placement-probe assignment is pseudo-randomized separately,
# then balanced by alternating earlier/later assignment among distance-2 pairs.
probe_assignments = init_placement_probe_assignments_for_block(PREDEFINED_PAIRS, rng)

example_rows = []

for mem_pos, pair in enumerate(ordered_pairs, start=1):
    pair_id_num = PREDEFINED_PAIRS.index(pair) + 1
    candidates = get_intervening_indices(pair)
    selected_probe = choose_placement_probe_index(pair, probe_assignments)

    if len(candidates) == 1:
        probe_position_label = "center"
    elif selected_probe == candidates[0]:
        probe_position_label = "earlier"
    elif selected_probe == candidates[1]:
        probe_position_label = "later"
    else:
        probe_position_label = "fallback"

    true_position_pct = (
        None if selected_probe is None
        else 100 * (selected_probe - pair[0]) / (pair[1] - pair[0])
    )

    example_rows.append(dict(
        memory_order_position=mem_pos,
        pair_id=f"P{pair_id_num:02d}",
        pair=f"({pair[0]}, {pair[1]})",
        endpoint_1=pair[0],
        endpoint_2=pair[1],
        candidate_indices=str(candidates),
        selected_probe_index=selected_probe,
        selected_probe_position=probe_position_label,
        true_position_pct=true_position_pct,
        design_role=placement_design_label(pair),
    ))

example_probe_df = pd.DataFrame(example_rows)

example_display_cols = [
    "memory_order_position",
    "pair_id",
    "pair",
    "candidate_indices",
    "selected_probe_index",
    "selected_probe_position",
    "true_position_pct",
    "design_role",
]

display(example_probe_df[example_display_cols])

# -----------------------------
# Balance check for this example block
# -----------------------------
balance_summary = (
    example_probe_df
    .groupby("selected_probe_position")
    .size()
    .reindex(["center", "earlier", "later"], fill_value=0)
    .rename("count")
    .reset_index()
)

display(balance_summary)

assert balance_summary.loc[
    balance_summary.selected_probe_position == "center", "count"
].iloc[0] == 6

assert balance_summary.loc[
    balance_summary.selected_probe_position == "earlier", "count"
].iloc[0] == 4

assert balance_summary.loc[
    balance_summary.selected_probe_position == "later", "count"
].iloc[0] == 4

print("✓ Example block has 6 center probes, 4 earlier probes, and 4 later probes.")
print("✓ Pair order is pseudo-randomized.")
print("✓ Distance-2 probe assignment is pseudo-randomized and balanced.")

# -----------------------------
# Publication-level visualization
# -----------------------------
fig, ax = plt.subplots(figsize=(16, 7.2))

# Change-point background lines
for cp in cp_list:
    ax.axvline(
        cp,
        color="#c9a227",
        ls="-",
        lw=1.3,
        alpha=0.35,
        zorder=1
    )

# Color by selected probe role.
role_colors = {
    "center": "#2f4b7c",   # single intervening candidate
    "earlier": "#a05195",  # earlier of two candidates
    "later": "#f95d6a",    # later of two candidates
}

role_labels = {
    "center": "single intervening item",
    "earlier": "earlier probe of the intervening two",
    "later": "later probe of the intervening two",
}

used_labels = set()

# Plot in memory-test order: y-axis is randomized order position.
for _, row in example_probe_df.iterrows():
    y = len(example_probe_df) - row.memory_order_position + 1
    ep1 = int(row.endpoint_1)
    ep2 = int(row.endpoint_2)
    probe = int(row.selected_probe_index)
    role = row.selected_probe_position

    color = role_colors.get(role, "#666666")
    label = role_labels.get(role, role)
    plot_label = label if label not in used_labels else None
    used_labels.add(label)

    # Span between endpoints
    ax.plot(
        [ep1, ep2],
        [y, y],
        color=color,
        lw=4.0,
        alpha=0.82,
        solid_capstyle="round",
        label=plot_label,
        zorder=3
    )

    # Endpoint items: circles
    ax.scatter(
        [ep1, ep2],
        [y, y],
        color=color,
        s=70,
        edgecolor="white",
        linewidth=0.8,
        zorder=4
    )

    # Candidate intervening items: faint vertical ticks
    candidates = get_intervening_indices([ep1, ep2])
    for c in candidates:
        ax.scatter(
            c,
            y,
            marker="|",
            color=color,
            s=230,
            lw=2.0,
            alpha=0.35,
            zorder=4
        )

    # Selected probe item: bold X
    ax.scatter(
        probe,
        y,
        marker="x",
        color="black",
        s=130,
        lw=2.8,
        zorder=6
    )

    # Left-side row label
    ax.text(
        -0.25,
        y,
        f"{row.memory_order_position:02d}  {row.pair_id}  {row.pair}",
        ha="right",
        va="center",
        fontsize=9,
        fontfamily="monospace"
    )

    # Right-side probe label
    ax.text(
        51.2,
        y,
        f"probe {probe} · {role} · {row.true_position_pct:.1f}%",
        ha="left",
        va="center",
        fontsize=8.5,
        color="#333333"
    )

# Custom legend elements
legend_handles = [
    Line2D([0], [0], color="#2f4b7c", lw=4, label="single intervening item"),
    Line2D([0], [0], color="#a05195", lw=4, label="earlier probe of two"),
    Line2D([0], [0], color="#f95d6a", lw=4, label="later probe of two"),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="#555555",
        markeredgecolor="white",
        markersize=8,
        label="endpoint item"
    ),
    Line2D(
        [0], [0],
        marker="|",
        color="#555555",
        linestyle="None",
        markersize=14,
        alpha=0.45,
        label="candidate intervening item"
    ),
    Line2D(
        [0], [0],
        marker="x",
        color="black",
        linestyle="None",
        markersize=9,
        markeredgewidth=2.5,
        label="selected slider probe"
    ),
    Line2D(
        [0], [0],
        color="#c9a227",
        lw=2,
        alpha=0.55,
        label=cp_label
    ),
]

ax.legend(
    handles=legend_handles,
    fontsize=9,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    frameon=False,
    ncol=4,
    handlelength=2.2,
    columnspacing=1.2
)

ax.set_xlim(0, 56)
ax.set_ylim(0.2, len(example_probe_df) + 0.8)
ax.set_xlabel("Trial serial position within block", fontsize=11)
ax.set_yticks([])
ax.set_ylabel("")

ax.set_title(
    "Memory-pair coverage and selected slider probes",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.text(
    0,
    len(example_probe_df) + 0.45,
    f"Example participant/block visualization · block {EXAMPLE_BLOCK} · seed {EXAMPLE_SEED}. "
    "Rows are randomized memory-test order; × marks the item actually placed on the slider.",
    fontsize=10.5,
    color="#444444"
)

# Light grid only on x-axis
ax.set_xticks(range(0, 51, 5))
ax.grid(axis="x", color="#dddddd", linewidth=0.8, alpha=0.55)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_alpha(0.35)

fig.tight_layout()

save_fig(fig, "memory_pair_coverage", also_html=True)

# Explicit HTML export columns so hidden/internal columns do not leak into reviewer-facing output.
save_html(
    "<h2>Memory Pair Candidate and Probe Structure</h2>"
    "<p>Every predefined memory pair receives a slider probe. "
    "For one-intervening-item pairs, the only intervening item is probed. "
    "For two-intervening-item pairs, the earlier/later probe assignment is pseudo-randomized "
    "and balanced within block.</p>"
    "<h3>Example participant/block probe assignment</h3>"
    + example_probe_df[example_display_cols].to_html(index=False)
    + "<h3>Probe balance for this example block</h3>"
    + balance_summary.to_html(index=False),
    "memory_pair_candidate_structure.html"
)

plt.show()

,memory_order_position,pair_id,pair,candidate_indices,selected_probe_index,selected_probe_position,true_position_pct,design_role
0,1,P06,"(17, 20)","[18, 19]",19,later,66.666667,earlier or later probe (counterbalanced)
1,2,P08,"(23, 26)","[24, 25]",25,later,66.666667,earlier or later probe (counterbalanced)
2,3,P03,"(7, 10)","[8, 9]",9,later,66.666667,earlier or later probe (counterbalanced)
3,4,P11,"(34, 36)",[35],35,center,50.000000,middle probe
4,5,P10,"(30, 33)","[31, 32]",31,earlier,33.333333,earlier or later probe (counterbalanced)
5,6,P01,"(2, 4)",[3],3,center,50.000000,middle probe
6,7,P04,"(11, 13)",[12],12,center,50.000000,middle probe
7,8,P05,"(16, 18)",[17],17,center,50.000000,middle probe
8,9,P02,"(6, 9)","[7, 8]",8,later,66.666667,earlier or later probe (counterbalanced)
9,10,P09,"(28, 31)","[29, 30]",29,earlier,33.333333,earlier or later probe (counterbalanced)


,selected_probe_position,count
0,center,6
1,earlier,4
2,later,4


✓ Example block has 6 center probes, 4 earlier probes, and 4 later probes.
✓ Pair order is pseudo-randomized.
✓ Distance-2 probe assignment is pseudo-randomized and balanced.
  ✓ Exported design_checks/memory_pair_coverage.png
  ✓ Exported design_checks/memory_pair_coverage.html
  ✓ Exported design_checks/memory_pair_candidate_structure_revised.html


## G. Slider-probe counterbalancing simulation

Every pair now gets a slider probe. Distance-1 pairs use the only candidate.
Distance-2 pairs balance earlier/later by shuffling the 8 distance-2 pairs
and alternating.

In [ ]:
# ============================================================
# Slider-probe counterbalancing simulation
# ============================================================
# Purpose:
#   Validate the stochastic part of the revised memory-test design.
#
#   For every simulated participant and every block:
#   - all 14 predefined memory pairs receive a slider probe
#   - 6 distance-1 pairs use the only intervening item ("center")
#   - 8 distance-2 pairs are split into 4 earlier probes and 4 later probes
#
#   Across simulated participants:
#   - earlier/later assignment for each distance-2 pair should be approximately balanced
#   - pair order should vary, but this section focuses on probe-balance, not pair-order distribution

NOTEBOOK_SEED = 42
N_SIM = 1000   # Use 50 for quick debugging; 1000+ is better for reviewer-facing stability.

rng_master = np.random.default_rng(NOTEBOOK_SEED)

# -----------------------------
# Helpers mirroring memory_task.js
# -----------------------------
def get_intervening(pair):
    """Return all serial positions between the two endpoint trials."""
    return list(range(pair[0] + 1, pair[1]))

def get_d2_pairs():
    """Distance-2 pairs have exactly two intervening candidate probes."""
    return [p for p in PREDEFINED_PAIRS if len(get_intervening(p)) == 2]

def pair_to_label(pair):
    """Stable pair label for tables/plots."""
    return f"P{PREDEFINED_PAIRS.index(pair) + 1:02d} {tuple(pair)}"

def simulate_block(block, rng):
    """
    Python mirror of memory_task.js placement logic.

    Runtime logic being validated:
    1. Memory-pair order is shuffled within block.
    2. Distance-2 pairs are separately shuffled.
    3. The shuffled distance-2 list is alternated earlier/later.
    4. Distance-1 pairs always use their only intervening item.
    """
    # Pair order pseudo-randomization
    order = PREDEFINED_PAIRS.copy()
    rng.shuffle(order)

    # Distance-2 probe assignment pseudo-randomization
    d2_pairs = get_d2_pairs()
    d2_shuffled = d2_pairs.copy()
    rng.shuffle(d2_shuffled)

    d2_assignment = {}
    for i, pair in enumerate(d2_shuffled):
        candidates = get_intervening(pair)
        choose_earlier = (i % 2 == 0)
        d2_assignment[tuple(pair)] = candidates[0] if choose_earlier else candidates[1]

    rows = []

    for mem_pos, pair in enumerate(order, start=1):
        candidates = get_intervening(pair)

        if len(candidates) == 1:
            probe_idx = candidates[0]
            probe_label = "center"
        elif len(candidates) == 2:
            probe_idx = d2_assignment[tuple(pair)]
            probe_label = "earlier" if probe_idx == candidates[0] else "later"
        else:
            # Current design should never reach this branch.
            probe_idx = candidates[0] if candidates else None
            probe_label = "fallback"

        true_pos_pct = (
            None if probe_idx is None
            else 100 * (probe_idx - pair[0]) / (pair[1] - pair[0])
        )

        rows.append(dict(
            block=block,
            memory_order_position=mem_pos,
            pair_id=f"P{PREDEFINED_PAIRS.index(pair) + 1:02d}",
            pair=tuple(pair),
            pair_label=pair_to_label(pair),
            endpoint_1=pair[0],
            endpoint_2=pair[1],
            candidate_indices=tuple(candidates),
            n_candidates=len(candidates),
            selected_probe_index=probe_idx,
            selected_probe_position=probe_label,
            selected_probe_ordinal=(candidates.index(probe_idx) + 1) if probe_idx in candidates else None,
            true_position_pct=round(true_pos_pct, 2) if true_pos_pct is not None else None,
        ))

    return rows

# -----------------------------
# Run simulation
# -----------------------------
all_rows = []

for sim_participant in range(N_SIM):
    # Separate participant-level RNG so each simulated participant is reproducible.
    rng = np.random.default_rng(NOTEBOOK_SEED + sim_participant)

    for block in range(1, 5):
        block_rows = simulate_block(block, rng)
        for row in block_rows:
            row["sim_participant"] = sim_participant
            all_rows.append(row)

sim_df = pd.DataFrame(all_rows)

# -----------------------------
# Hard invariant checks
# -----------------------------
for sim_participant in range(N_SIM):
    for block in range(1, 5):
        bdf = sim_df[
            (sim_df.sim_participant == sim_participant) &
            (sim_df.block == block)
        ]

        assert len(bdf) == 14
        assert (bdf.selected_probe_position == "center").sum() == 6
        assert (bdf.selected_probe_position == "earlier").sum() == 4
        assert (bdf.selected_probe_position == "later").sum() == 4
        assert bdf.pair_id.nunique() == 14

print(f"✓ {N_SIM} simulated participants × 4 blocks checked.")
print("✓ Every participant-block has 14 slider trials.")
print("✓ Every participant-block has exact probe balance: 6 center / 4 earlier / 4 later.")
print("✓ Every pair appears once per participant-block.")

# -----------------------------
# Summary tables
# -----------------------------
participant_block_summary = (
    sim_df
    .groupby(["sim_participant", "block", "selected_probe_position"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["center", "earlier", "later"]:
    if col not in participant_block_summary.columns:
        participant_block_summary[col] = 0

summary_by_block = (
    participant_block_summary
    .groupby("block")[["center", "earlier", "later"]]
    .agg(["mean", "min", "max"])
)

display(summary_by_block)

# Distance-2 pair-level assignment rate across simulations.
d2_rate_df = (
    sim_df[sim_df.n_candidates == 2]
    .groupby(["block", "pair_id", "pair_label", "selected_probe_position"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["earlier", "later"]:
    if col not in d2_rate_df.columns:
        d2_rate_df[col] = 0

d2_rate_df["total"] = d2_rate_df["earlier"] + d2_rate_df["later"]
d2_rate_df["p_earlier"] = d2_rate_df["earlier"] / d2_rate_df["total"]
d2_rate_df["p_later"] = d2_rate_df["later"] / d2_rate_df["total"]
d2_rate_df["earlier_minus_later"] = d2_rate_df["earlier"] - d2_rate_df["later"]

display(d2_rate_df[[
    "block",
    "pair_id",
    "pair_label",
    "earlier",
    "later",
    "total",
    "p_earlier",
    "p_later",
    "earlier_minus_later",
]])

# ============================================================
# Plot 1: exact within-block balance
# ============================================================
balance_long = (
    participant_block_summary
    .melt(
        id_vars=["sim_participant", "block"],
        value_vars=["center", "earlier", "later"],
        var_name="probe_role",
        value_name="count"
    )
)

plot_summary = (
    balance_long
    .groupby(["block", "probe_role"])["count"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

role_order = ["center", "earlier", "later"]
role_colors = {
    "center": "#2f4b7c",
    "earlier": "#a05195",
    "later": "#f95d6a",
}
role_labels = {
    "center": "single intervening item",
    "earlier": "earlier of two intervening items",
    "later": "later of two intervening items",
}

fig, ax = plt.subplots(figsize=(10.5, 5.8))

x = np.arange(1, 5)
bottom = np.zeros(len(x))

for role in role_order:
    vals = (
        plot_summary[plot_summary.probe_role == role]
        .sort_values("block")["mean"]
        .values
    )

    ax.bar(
        x,
        vals,
        bottom=bottom,
        color=role_colors[role],
        edgecolor="white",
        linewidth=1.0,
        label=role_labels[role],
        width=0.68,
    )

    # Segment labels
    for xi, btm, val in zip(x, bottom, vals):
        ax.text(
            xi,
            btm + val / 2,
            f"{int(val)}",
            ha="center",
            va="center",
            color="white",
            fontsize=11,
            fontweight="bold"
        )

    bottom += vals

# Total labels
for xi, total in zip(x, bottom):
    ax.text(
        xi,
        total + 0.35,
        f"{int(total)} total",
        ha="center",
        va="bottom",
        fontsize=10,
        color="#333333"
    )

ax.set_xticks(x)
ax.set_xticklabels([f"Block {i}" for i in x])
ax.set_ylim(0, 16)
ax.set_ylabel("Slider-probe trials per block", fontsize=11)
ax.set_title(
    "Exact within-block slider-probe balance",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.text(
    0.02,
    0.96,
    f"N = {N_SIM} simulated participants. Each participant-block is constrained to 6 center, 4 earlier, and 4 later probes.",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    color="#444444"
)

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
    fontsize=9
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#dddddd", linewidth=0.8, alpha=0.45)
ax.set_axisbelow(True)

fig.tight_layout()
save_fig(fig, "slider_probe_counterbalance_exact_within_block", also_html=True)
plt.show()

# ============================================================
# Plot 2: pair-level earlier-assignment probability
# ============================================================
# This asks a different question:
#   Across simulated participants, is each distance-2 pair approximately
#   equally likely to be assigned as the earlier vs later probe?

heat = (
    d2_rate_df
    .pivot(index="block", columns="pair_label", values="p_earlier")
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14.5, 4.8))

im = ax.imshow(
    heat.values,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="RdBu_r"
)

ax.set_xticks(np.arange(heat.shape[1]))
ax.set_xticklabels(heat.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(np.arange(heat.shape[0]))
ax.set_yticklabels([f"Block {b}" for b in heat.index], fontsize=10)

# Annotate cells with p(earlier)
for r in range(heat.shape[0]):
    for c in range(heat.shape[1]):
        val = heat.values[r, c]
        ax.text(
            c,
            r,
            f"{val:.2f}",
            ha="center",
            va="center",
            fontsize=8.5,
            color="white" if val < 0.30 or val > 0.70 else "#222222"
        )

cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.015)
cbar.set_label("Probability of earlier-probe assignment", fontsize=10)

ax.set_title(
    "Pair-level earlier/later assignment across simulated participants",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Distance-2 memory pair", fontsize=11)
ax.set_ylabel("Block", fontsize=11)

fig.tight_layout()
save_fig(fig, "slider_probe_pair_level_assignment_probability", also_html=True)
plt.show()

# ============================================================
# Reviewer-facing HTML summary
# ============================================================
uid = "slider_probe_counterbalance"

summary_html = (
    styled_table_css(uid)
    + f'<div id="{uid}">'
    + "<h2>Slider-Probe Counterbalancing Simulation</h2>"
    + "<p>This simulation mirrors the revised memory-task placement logic: "
      "every memory pair receives a slider probe; one-intervening-item pairs use the only candidate; "
      "two-intervening-item pairs are pseudo-randomized and assigned with exact within-block "
      "earlier/later balance.</p>"
    + f"<p><strong>Simulation:</strong> {N_SIM} participants × 4 blocks.</p>"
    + "<h3>Participant-block invariant summary</h3>"
    + summary_by_block.to_html()
    + "<h3>Distance-2 pair-level assignment rates</h3>"
    + d2_rate_df[[
        "block",
        "pair_id",
        "pair_label",
        "earlier",
        "later",
        "total",
        "p_earlier",
        "p_later",
        "earlier_minus_later",
      ]].to_html(index=False)
    + "</div>"
)

save_html(summary_html, "slider_probe_counterbalance_summary.html")

✓ 1000 simulated participants × 4 blocks checked.
✓ Every participant-block has 14 slider trials.
✓ Every participant-block has exact probe balance: 6 center / 4 earlier / 4 later.
✓ Every pair appears once per participant-block.


selected_probe_position center         earlier         later        
                          mean min max    mean min max  mean min max
block                                                               
1                          6.0   6   6     4.0   4   4   4.0   4   4
2                          6.0   6   6     4.0   4   4   4.0   4   4
3                          6.0   6   6     4.0   4   4   4.0   4   4
4                          6.0   6   6     4.0   4   4   4.0   4   4

selected_probe_position,block,pair_id,pair_label,earlier,later,total,p_earlier,p_later,earlier_minus_later
0,1,P02,"P02 (6, 9)",520,480,1000,0.520,0.480,40
1,1,P03,"P03 (7, 10)",514,486,1000,0.514,0.486,28
2,1,P06,"P06 (17, 20)",497,503,1000,0.497,0.503,-6
3,1,P08,"P08 (23, 26)",495,505,1000,0.495,0.505,-10
4,1,P09,"P09 (28, 31)",502,498,1000,0.502,0.498,4
5,1,P10,"P10 (30, 33)",495,505,1000,0.495,0.505,-10
6,1,P12,"P12 (35, 38)",502,498,1000,0.502,0.498,4
7,1,P14,"P14 (42, 45)",475,525,1000,0.475,0.525,-50
8,2,P02,"P02 (6, 9)",504,496,1000,0.504,0.496,8
9,2,P03,"P03 (7, 10)",495,505,1000,0.495,0.505,-10


  ✓ Exported design_checks/slider_probe_counterbalance_exact_within_block.png
  ✓ Exported design_checks/slider_probe_counterbalance_exact_within_block.html
  ✓ Exported design_checks/slider_probe_pair_level_assignment_probability.png
  ✓ Exported design_checks/slider_probe_pair_level_assignment_probability.html
  ✓ Exported design_checks/slider_probe_counterbalance_summary.html


## H. Pair-order pseudo-randomization simulation

This check validates the randomization of pair order, independent of the slider probe assignment.
> Is memory-test order randomized enough that a given pair is not always tested early or late?

In [29]:
N_RAND = 1000
pos_counts = {blk: {str(p): Counter() for p in PREDEFINED_PAIRS} for blk in range(1,5)}
for pid in range(N_RAND):
    rng = np.random.default_rng(NOTEBOOK_SEED*1000+pid)
    for blk in range(1,5):
        order = PREDEFINED_PAIRS.copy(); rng.shuffle(order)
        seen = set()
        for pos, pair in enumerate(order):
            k = tuple(pair); assert k not in seen; seen.add(k)
            pos_counts[blk][str(pair)][pos] += 1

pos_dist = pd.DataFrame(pos_counts[1]).T.fillna(0).astype(int)
pos_dist.columns = [f"pos_{c}" for c in pos_dist.columns]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pos_dist.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(pos_dist.shape[1]))
ax.set_xticklabels(pos_dist.columns, fontsize=8)
ax.set_yticks(range(pos_dist.shape[0]))
ax.set_yticklabels(pos_dist.index, fontsize=9, fontfamily="monospace")
ax.set_xlabel("Memory-test pair"); ax.set_ylabel("Pair")
ax.set_title(f"Pair \u00d7 Position counts (Block 1, N={N_RAND})", fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.7, label="Count")
for r in range(pos_dist.shape[0]):
    for c in range(pos_dist.shape[1]):
        v = pos_dist.values[r,c]
        ax.text(c, r, str(v), ha="center", va="center", fontsize=7,
                color="white" if v > N_RAND/14*1.3 else "#333")
fig.tight_layout()
save_fig(fig, "pair_order_pseudorandomization", also_html=True)
plt.show()
print(f"\u2713 {N_RAND} sims: all pairs once per block, broad position coverage.")

  ✓ Exported design_checks/pair_order_pseudorandomization.png
  ✓ Exported design_checks/pair_order_pseudorandomization.html
✓ 1000 sims: all pairs once per block, broad position coverage.


## J. Image assignment to serial positions
Adapts the emoji-ribbon visualization to PNG-based stimuli,
showing image names instead of emoji characters in a color-coded block ribbon.

### Shared thumbnail helpers

In [32]:
from pathlib import Path
import base64
from io import BytesIO
import html
import textwrap
import numpy as np
import pandas as pd

try:
    from PIL import Image, ImageOps
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False
    print("[warning] Pillow is not installed. HTML will fall back to filenames instead of thumbnails.")

REPO_ROOT = Path.cwd()
STIM_DIR = REPO_ROOT / "static" / "stimuli"

# Retina-style thumbnail settings:
# encode_px = actual embedded PNG size
# display_px = CSS display size
THUMB_ENCODE_PX = 88
THUMB_DISPLAY_PX = 44

RIBBON_ENCODE_PX = 76
RIBBON_DISPLAY_PX = 38

PROBE_ENCODE_PX = 96
PROBE_DISPLAY_PX = 48

_thumb_cache = {}

def stimulus_path_to_abs(path):
    if path is None or pd.isna(path):
        return None
    return REPO_ROOT / str(path)

def stimulus_name_from_path(path):
    if path is None or pd.isna(path):
        return "?"
    return (
        str(path)
        .replace("static/stimuli/", "")
        .replace(".png", "")
    )

def img_to_base64_thumb(path, encode_px=88, canvas_px=None):
    """
    Return a crisp base64 PNG thumbnail data URI.

    encode_px controls actual embedded resolution.
    display size is controlled separately in CSS.

    canvas_px gives all thumbnails a stable square canvas, which keeps
    tables/ribbons aligned even when source PNGs have different aspect ratios.
    """
    if not PIL_AVAILABLE:
        return None

    if path is None or pd.isna(path):
        return None

    if canvas_px is None:
        canvas_px = encode_px

    cache_key = (str(path), int(encode_px), int(canvas_px))
    if cache_key in _thumb_cache:
        return _thumb_cache[cache_key]

    p = stimulus_path_to_abs(path)

    if p is None or not p.exists():
        _thumb_cache[cache_key] = None
        return None

    try:
        im = Image.open(p).convert("RGBA")

        # Fit inside a square canvas without distortion.
        im.thumbnail((encode_px, encode_px), Image.LANCZOS)

        canvas = Image.new("RGBA", (canvas_px, canvas_px), (255, 255, 255, 0))
        x = (canvas_px - im.width) // 2
        y = (canvas_px - im.height) // 2
        canvas.alpha_composite(im, (x, y))

        buf = BytesIO()
        canvas.save(buf, format="PNG", optimize=True)
        encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
        uri = f"data:image/png;base64,{encoded}"

        _thumb_cache[cache_key] = uri
        return uri

    except Exception as e:
        print(f"[thumbnail warning] Could not render {p}: {e}")
        _thumb_cache[cache_key] = None
        return None

def stim_thumb_html(path, name=None, encode_px=88, display_px=44, show_name=True, css_class="stim-thumb"):
    """
    HTML thumbnail. Uses a 2x-ish embedded PNG for sharper CSS display.
    """
    if name is None:
        name = stimulus_name_from_path(path)

    safe_name = html.escape(str(name))
    uri = img_to_base64_thumb(path, encode_px=encode_px, canvas_px=encode_px)

    if uri is None:
        return f'<div class="stim-fallback">{safe_name}</div>'

    label_html = f'<div class="stim-name">{safe_name}</div>' if show_name else ""

    return (
        f'<div class="stim-thumb-wrap" style="min-width:{display_px + 14}px;">'
        f'<img class="{css_class}" src="{uri}" alt="{safe_name}" title="{safe_name}" '
        f'style="width:{display_px}px;height:{display_px}px;"/>'
        f'{label_html}'
        f'</div>'
    )

def get_row_value(row, *names, default=None):
    for name in names:
        if name in row.index:
            return row[name]
    return default

def pair_from_row_value(x):
    if isinstance(x, (tuple, list)):
        return tuple(int(v) for v in x)

    s = str(x).strip()
    s = s.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return tuple(int(p) for p in parts)

def gimg_from_block(blk_imgs, trial_num):
    row = blk_imgs[blk_imgs.trial == int(trial_num)]
    if len(row) == 0:
        return {"path": None, "name": "?"}

    return {
        "path": row.iloc[0]["path"],
        "name": row.iloc[0]["name"],
    }

thumb_css = textwrap.dedent(f"""\
<style>
.stim-table-doc {{
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
  color: #222;
  background: #fff;
  padding: 18px 12px;
}}
.stim-table-doc h1,
.stim-table-doc h2,
.stim-table-doc h3 {{
  margin: 0 0 12px 0;
}}
.stim-table-doc p {{
  max-width: 980px;
  font-size: 13px;
  line-height: 1.45;
  color: #475569;
}}
.stim-table-doc table {{
  border-collapse: collapse;
  font-size: 12px;
  width: 100%;
  max-width: 1500px;
}}
.stim-table-doc caption {{
  caption-side: top;
  text-align: left;
  font-weight: 750;
  padding: 0 0 8px 0;
}}
.stim-table-doc th {{
  background: #f4f6f8;
  border: 1px solid #d0d7de;
  padding: 6px 8px;
  text-align: left;
  font-weight: 750;
}}
.stim-table-doc td {{
  border: 1px solid #d0d7de;
  padding: 5px 7px;
  vertical-align: middle;
}}
.stim-thumb-wrap {{
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 2px;
}}
.stim-thumb {{
  object-fit: contain;
  border-radius: 8px;
  background: #f8fafc;
  border: 1px solid #d8dee9;
  padding: 3px;
  image-rendering: auto;
}}
.stim-name {{
  font-size: 9px;
  max-width: 88px;
  line-height: 1.05;
  text-align: center;
  color: #334155;
  word-break: break-word;
}}
.stim-fallback {{
  font-size: 10px;
  max-width: 90px;
  word-break: break-word;
  color: #475569;
}}
</style>
""")

# -----------------------------
# Simulate one participant-level image assignment
# -----------------------------
assert len(STIMULUS_IMAGE_POOL) >= 200, "STIMULUS_IMAGE_POOL must contain at least 200 paths."

rng_j = np.random.default_rng(NOTEBOOK_SEED)
shuffled_pool = STIMULUS_IMAGE_POOL.copy()
rng_j.shuffle(shuffled_pool)

img_rows = []

for blk in range(4):
    for t in range(50):
        gi = blk * 50 + t
        path = shuffled_pool[gi]
        abs_path = stimulus_path_to_abs(path)

        img_rows.append(dict(
            global_trial=gi + 1,
            block=blk + 1,
            trial=t + 1,
            path=path,
            name=stimulus_name_from_path(path),
            file_exists=abs_path.exists() if abs_path else False,
        ))

img_df = pd.DataFrame(img_rows)

display(img_df.head(20))

assert len(img_df) == 200
assert img_df["path"].nunique() == 200
assert img_df["file_exists"].all(), "Some stimulus PNG files are missing under static/stimuli/."

print("✓ Shared image-assignment setup complete.")
print("✓ 200 unique PNG paths assigned to 4 blocks × 50 trials.")

,global_trial,block,trial,path,name,file_exists
0,1,1,1,static/stimuli/shield.png,shield,True
1,2,1,2,static/stimuli/bowling.png,bowling,True
2,3,1,3,static/stimuli/newspaper.png,newspaper,True
3,4,1,4,static/stimuli/clutch.png,clutch,True
4,5,1,5,static/stimuli/carp-streamer.png,carp-streamer,True
5,6,1,6,static/stimuli/auto-rickshaw.png,auto-rickshaw,True
6,7,1,7,static/stimuli/goggles.png,goggles,True
7,8,1,8,static/stimuli/sunflower.png,sunflower,True
8,9,1,9,static/stimuli/saxophone.png,saxophone,True
9,10,1,10,static/stimuli/crown.png,crown,True


✓ Shared image-assignment setup complete.
✓ 200 unique PNG paths assigned to 4 blocks × 50 trials.


### Stimulus ribbons html

Example participant stimulus ribbon under a fixed notebook seed.

What is universal across participants:
- block structure
- memory-pair endpoint positions
- candidate probe positions
- change-point positions

What varies across participants:
- which PNG object appears at trial 1, trial 2, ..., trial 200

In [33]:
# ============================================================
# Export 1: stimulus_ribbons.html
# ============================================================

ribbon_css = textwrap.dedent(f"""\
<style>
.ribbon-doc {{
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
  color: #222;
  background: #fff;
  padding: 18px 8px;
}}
.ribbon-doc h1 {{
  font-size: 18px;
  font-weight: 750;
  margin: 0 0 8px 0;
}}
.ribbon-doc .note {{
  max-width: 980px;
  font-size: 12px;
  line-height: 1.45;
  color: #475569;
  margin: 0 0 12px 0;
}}
.ribbon-doc .legend {{
  display: flex;
  flex-wrap: wrap;
  gap: 6px 14px;
  align-items: center;
  font-size: 11.5px;
  color: #333;
  margin: 8px 0 14px 0;
}}
.ribbon-doc .legend-item {{
  display: inline-flex;
  align-items: center;
  gap: 5px;
  white-space: nowrap;
}}
.ribbon-doc .swatch {{
  display: inline-block;
  width: 16px;
  height: 12px;
  border-radius: 2px;
}}
.ribbon-doc .swatch.tested {{
  background: #fff3a3;
  border: 1px solid #b58a00;
}}
.ribbon-doc .swatch.probe {{
  background: #cfe8ff;
  border: 1px solid #2b6cb0;
}}
.ribbon-doc .swatch.cp-line {{
  width: 3px;
  height: 16px;
  background: #d62728;
}}
.ribbon-doc .ribbon-title {{
  font-weight: 700;
  font-size: 12.5px;
  margin: 14px 0 4px 0;
}}
.ribbon-doc table.ribbon {{
  border-collapse: collapse;
  table-layout: fixed;
  width: 100%;
  max-width: 1650px;
}}
.ribbon-doc table.ribbon td {{
  width: 2%;
  height: 68px;
  border: 1px solid #d2d2d2;
  text-align: center;
  vertical-align: middle;
  position: relative;
  background: #fff;
  overflow: hidden;
  padding: 2px;
}}
.ribbon-doc table.ribbon td.tested {{
  background: #fff3a3;
  border-color: #b58a00;
}}
.ribbon-doc table.ribbon td.probe {{
  background: #cfe8ff;
  border-color: #2b6cb0;
}}
.ribbon-doc table.ribbon td.cp-after {{
  border-right: 3px solid #d62728;
}}
.ribbon-doc .pair-label {{
  position: absolute;
  top: 1px;
  left: 1px;
  font-size: 7px;
  font-weight: 800;
  color: #5b4400;
  background: rgba(255,255,255,.88);
  border-radius: 2px;
  padding: 0 2px;
}}
.ribbon-doc .probe-label {{
  position: absolute;
  top: 1px;
  right: 1px;
  font-size: 7px;
  font-weight: 800;
  color: #004b88;
  background: rgba(255,255,255,.88);
  border-radius: 2px;
  padding: 0 2px;
}}
.ribbon-doc .trial-num {{
  position: absolute;
  bottom: 1px;
  right: 2px;
  font-size: 7px;
  color: #777;
}}
.ribbon-doc .ribbon-img {{
  width: {RIBBON_DISPLAY_PX}px;
  height: {RIBBON_DISPLAY_PX}px;
  object-fit: contain;
  display: block;
  margin: 15px auto 0 auto;
  image-rendering: auto;
}}
.ribbon-doc .missing {{
  font-size: 8px;
  color: #64748b;
  margin-top: 24px;
}}
</style>
""")

# Build pair endpoint lookup and candidate/probe lookup.
pair_endpoints = set()
pair_labels = {}
candidate_positions = set()

for pi, pair in enumerate(PREDEFINED_PAIRS):
    ep1, ep2 = pair

    for ep in [ep1, ep2]:
        pair_endpoints.add(ep)
        pair_labels.setdefault(ep, []).append(f"P{pi+1:02d}")

    for c in range(ep1 + 1, ep2):
        candidate_positions.add(c)

ribbon_body = [
    '<div class="ribbon-doc">',
    '<h1>DroneTask stimulus</h1>',
    '<p class="note">Each cell is one encoding trial. Yellow marks memory-pair endpoints; blue marks slider-probe positions; red borders mark change-point boundaries. </p>',
    '<div class="legend">',
    '<span class="legend-item"><span class="swatch tested"></span>memory-pair endpoint item</span>',
    '<span class="legend-item"><span class="swatch probe"></span>slider-probe item</span>',
    '<span class="legend-item"><span class="swatch cp-line"></span>change-point boundary</span>',
    '</div>'
]

for blk in range(1, 5):
    vol = int(factors_vol[blk - 1])
    cps = set(CP_HIGH_VOL if vol == 49 else CP_LOW_VOL)
    blk_imgs = img_df[img_df.block == blk].sort_values("trial")

    ribbon_body.append(
        f'<div class="ribbon-title">Block {blk} '
        f'({html.escape(str(factors_valence[blk - 1]))}, '
        f'{"high" if vol == 49 else "low"} volatility)</div>'
    )

    ribbon_body.append('<table class="ribbon"><tr>')

    for t in range(1, 51):
        classes = []
        labels = ""

        if t in pair_endpoints:
            classes.append("tested")

        if t in candidate_positions:
            classes.append("probe")

        if t in cps:
            classes.append("cp-after")

        if t in pair_labels:
            labels += f'<span class="pair-label">{html.escape(",".join(pair_labels[t]))}</span>'

        if t in candidate_positions:
            labels += '<span class="probe-label">PRB</span>'

        row_t = blk_imgs[blk_imgs.trial == t]

        if len(row_t) > 0:
            path = row_t.iloc[0]["path"]
            name = row_t.iloc[0]["name"]
            uri = img_to_base64_thumb(path, encode_px=RIBBON_ENCODE_PX, canvas_px=RIBBON_ENCODE_PX)
        else:
            name, uri = "?", None

        safe_name = html.escape(str(name))
        cls = f' class="{" ".join(classes)}"' if classes else ""

        if uri:
            stim_content = (
                f'<img class="ribbon-img" src="{uri}" alt="{safe_name}" title="{safe_name}"/>'
            )
        else:
            stim_content = f'<div class="missing">{safe_name}</div>'

        ribbon_body.append(
            f'<td{cls}>{labels}{stim_content}<span class="trial-num">{t}</span></td>'
        )

    ribbon_body.append("</tr></table>")

ribbon_body.append("</div>")

save_html(
    ribbon_css + "\n".join(ribbon_body),
    "stimulus_ribbons.html"
)

print("✓ Exported stimulus_ribbons.html.")

  ✓ Exported design_checks/stimulus_ribbons.html
✓ Exported stimulus_ribbons.html.


### Items to serial position example

Item assignment to serial positions for one simulated participant. The actual task shuffles STIMULUS_IMAGE_POOL per participant.

In [35]:
# ============================================================
# Export 2: items_to_serial_positions.html
# ============================================================

imgassign_rows = []

for _, r in img_df.head(50).iterrows():
    imgassign_rows.append(
        f"<tr>"
        f"<td>{int(r.global_trial)}</td>"
        f"<td>{int(r.block)}</td>"
        f"<td>{int(r.trial)}</td>"
        f"<td>{stim_thumb_html(r.path, r['name'], encode_px=THUMB_ENCODE_PX, display_px=THUMB_DISPLAY_PX, show_name=False)}</td>"
        f"<td>{html.escape(str(r['name']))}</td>"
        f"<td>{html.escape(str(r.path))}</td>"
        f"</tr>"
    )

save_html(
    thumb_css
    + styled_table_css("imgassign")
    + '<div id="imgassign" class="stim-table-doc">'
    + "<h2>Items assignment to serial positions</h2>"
    + "<table>"
    + "<caption>Items to serial positions: first block / first 50 rows</caption>"
    + "<tr><th>global_trial</th><th>block</th><th>within_block_trial</th><th>thumbnail</th><th>name</th><th>path</th></tr>"
    + "".join(imgassign_rows)
    + "</table></div>",
    "items_to_serial_positions.html"
)

display(img_df.head(50)[["global_trial", "block", "trial", "name", "path", "file_exists"]])

print("✓ Exported items_to_serial_positions.html.")

  ✓ Exported design_checks/items_to_serial_positions.html


,global_trial,block,trial,name,path,file_exists
0,1,1,1,shield,static/stimuli/shield.png,True
1,2,1,2,bowling,static/stimuli/bowling.png,True
2,3,1,3,newspaper,static/stimuli/newspaper.png,True
3,4,1,4,clutch,static/stimuli/clutch.png,True
4,5,1,5,carp-streamer,static/stimuli/carp-streamer.png,True
5,6,1,6,auto-rickshaw,static/stimuli/auto-rickshaw.png,True
6,7,1,7,goggles,static/stimuli/goggles.png,True
7,8,1,8,sunflower,static/stimuli/sunflower.png,True
8,9,1,9,saxophone,static/stimuli/saxophone.png,True
9,10,1,10,crown,static/stimuli/crown.png,True


✓ Exported items_to_serial_positions.html.


### memory_probe_assignment.html

The pair structure is universal, but the specific endpoint/probe items and the earlier/later probe item for the distance-2 pairs vary across participants.

In [36]:
# ============================================================
# Export 3: memory_probe_image_assignment.html
# ============================================================

required_sim_cols_any = (
    ("selected_probe_index" in sim_df.columns) or ("probe_idx" in sim_df.columns)
)
assert required_sim_cols_any, "sim_df must include selected_probe_index or probe_idx from the counterbalancing simulation cell."

probe_img_rows = []

for blk in range(1, 5):
    blk_imgs = img_df[img_df.block == blk].sort_values("trial")

    p0_blk = (
        sim_df[(sim_df.sim_participant == 0) & (sim_df.block == blk)]
        .copy()
        .sort_values("pair_id")
    )

    for _, r in p0_blk.iterrows():
        pair = pair_from_row_value(r["pair"])
        ep1, ep2 = pair

        probe_idx = get_row_value(r, "selected_probe_index", "probe_idx")
        probe_label = get_row_value(r, "selected_probe_position", "probe_label", default="?")

        first = gimg_from_block(blk_imgs, ep1)
        probe = gimg_from_block(blk_imgs, probe_idx)
        second = gimg_from_block(blk_imgs, ep2)

        probe_img_rows.append(dict(
            block=blk,
            pair_id=r.pair_id,
            pair=str(pair),
            ep1_idx=ep1,
            ep1_img=first["name"],
            ep1_path=first["path"],
            probe_idx=int(probe_idx),
            probe_img=probe["name"],
            probe_path=probe["path"],
            ep2_idx=ep2,
            ep2_img=second["name"],
            ep2_path=second["path"],
            probe_label=probe_label,
        ))

probe_img_df = pd.DataFrame(probe_img_rows)

probe_display_cols = [
    "block",
    "pair_id",
    "pair",
    "ep1_idx",
    "ep1_img",
    "probe_idx",
    "probe_img",
    "ep2_idx",
    "ep2_img",
    "probe_label",
]

display(probe_img_df[probe_display_cols])

probe_img_html_rows = []

for _, row in probe_img_df.iterrows():
    probe_img_html_rows.append(
        f"<tr>"
        f"<td>{row['block']}</td>"
        f"<td>{html.escape(str(row['pair_id']))}</td>"
        f"<td>{html.escape(str(row['pair']))}</td>"
        f"<td>{row['ep1_idx']}</td>"
        f"<td>{stim_thumb_html(row['ep1_path'], row['ep1_img'], encode_px=PROBE_ENCODE_PX, display_px=PROBE_DISPLAY_PX, show_name=True)}</td>"
        f"<td>{row['probe_idx']}</td>"
        f"<td>{stim_thumb_html(row['probe_path'], row['probe_img'], encode_px=PROBE_ENCODE_PX, display_px=PROBE_DISPLAY_PX, show_name=True)}</td>"
        f"<td>{row['ep2_idx']}</td>"
        f"<td>{stim_thumb_html(row['ep2_path'], row['ep2_img'], encode_px=PROBE_ENCODE_PX, display_px=PROBE_DISPLAY_PX, show_name=True)}</td>"
        f"<td>{html.escape(str(row['probe_label']))}</td>"
        f"</tr>"
    )

save_html(
    thumb_css
    + styled_table_css("probeimg")
    + '<div id="probeimg" class="stim-table-doc">'
    + "<h2>Memory probe assignment</h2>"
    + "<p>Each row shows the two endpoint items and the selected slider-probe item for one simulated participant. </p>"
    + "<table>"
    + "<caption>Memory probe assignment</caption>"
    + "<tr>"
    + "<th>block</th><th>pair_id</th><th>pair</th>"
    + "<th>endpoint 1 index</th><th>endpoint 1 item</th>"
    + "<th>probe index</th><th>selected probe item</th>"
    + "<th>endpoint 2 index</th><th>endpoint 2 item</th>"
    + "<th>probe role</th>"
    + "</tr>"
    + "".join(probe_img_html_rows)
    + "</table></div>",
    "memory_probe_assignment.html"
)

print("✓ Exported memory_probe_assignment.html.")

,block,pair_id,pair,ep1_idx,ep1_img,probe_idx,probe_img,ep2_idx,ep2_img,probe_label
0,1,P01,"(2, 4)",2,bowling,3,newspaper,4,clutch,center
1,1,P02,"(6, 9)",6,auto-rickshaw,7,goggles,9,saxophone,earlier
2,1,P03,"(7, 10)",7,goggles,9,saxophone,10,crown,later
3,1,P04,"(11, 13)",11,black-nib,12,telescope,13,microscope,center
4,1,P05,"(16, 18)",16,yarn,17,watch,18,card-file-box,center
5,1,P06,"(17, 20)",17,watch,19,robot,20,computer-disk,later
6,1,P07,"(22, 24)",22,violin,23,sled,24,popcorn,center
7,1,P08,"(23, 26)",23,sled,25,video-camera,26,shopping-bags,later
8,1,P09,"(28, 31)",28,wrapped-gift,29,satellite,31,funeral-urn,earlier
9,1,P10,"(30, 33)",30,spiral-shell,32,fire,33,mate,later


  ✓ Exported design_checks/memory_probe_assignment.html
✓ Exported memory_probe_assignment.html.


## K. Drop-object distribution and feedback audit

In [13]:
src_drop = details_src if details_src else stimuli_src
drop_dist = parse_js_flat_array(src_drop, "drop_obj_distribution_default")
drop_dur  = parse_js_flat_array(src_drop, "drop_obj_duration_default")

drop_df = pd.DataFrame({"fragment": range(len(drop_dist)),
    "h_offset": drop_dist, "duration_ms": [int(d) for d in drop_dur]})
display(drop_df)

assert len(drop_dist)==10 and len(drop_dur)==10
assert all(np.isfinite(drop_dist)) and all(np.isfinite(drop_dur))
print("\u2713 10 fragments, matching lengths, all finite.")

fb_df = pd.DataFrame({"captured": range(11),
    "reward_fb": range(11), "loss_fb": [c-10 for c in range(11)]})
display(fb_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.bar(drop_df.fragment, drop_df.h_offset, color="#1f77b4", alpha=0.8)
ax1.set_xlabel("Fragment index"); ax1.set_ylabel("Horizontal offset (pp)")
ax1.set_title("Drop-object horizontal distribution", fontweight="bold")
ax2.bar(drop_df.fragment, drop_df.duration_ms, color="#ff7f0e", alpha=0.8)
ax2.set_xlabel("Fragment index"); ax2.set_ylabel("Duration (ms)")
ax2.set_title("Drop-object fall durations", fontweight="bold")
fig.tight_layout()
save_fig(fig, "drop_object_distribution", also_html=True)
plt.show()

save_html(
    styled_table_css("dropfb") + '<div id="dropfb">' +
    "<table><caption>Drop-object distribution</caption>" + drop_df.to_html(index=False) + "</table>" +
    "<table><caption>Reward / loss feedback mapping</caption>" + fb_df.to_html(index=False) + "</table></div>",
    "drop_object_distribution_feedback_audit.html")

,fragment,h_offset,duration_ms
0,0,-4.25,350
1,1,-3.00,400
2,2,-1.75,500
3,3,-0.75,550
4,4,-0.25,600
5,5,0.25,600
6,6,0.75,600
7,7,1.75,500
8,8,3.00,500
9,9,4.25,400


✓ 10 fragments, matching lengths, all finite.


,captured,reward_fb,loss_fb
0,0,0,-10
1,1,1,-9
2,2,2,-8
3,3,3,-7
4,4,4,-6
5,5,5,-5
6,6,6,-4
7,7,7,-3
8,8,8,-2
9,9,9,-1


  ✓ Exported design_checks/drop_object_distribution.png
  ✓ Exported design_checks/drop_object_distribution.html
  ✓ Exported design_checks/drop_object_distribution_feedback_audit.html


## L. Expected data dictionary / CSV columns

In [14]:
dict_rows = [
    ("placement_probe_index", "int", "1-indexed within-block serial position of the selected slider probe item."),
    ("placement_probe_img", "str", "Stimulus path (PNG) or emoji string of the selected probe."),
    ("placement_probe_position", "int", "Ordinal among candidates (1 = earliest)."),
    ("placement_candidate_indices", "str", "Comma-separated 1-indexed positions of all candidate items."),
    ("placement_num_candidate_items", "int", "Number of candidates (1 or 2)."),
    ("placement_trial_type", "str", "Detailed label: legacy pair status + distance + probe position."),
    ("placement_true_position_pct", "float", "True serial position on slider (0\u2013100). E.g. [2,4] probe 3 \u2192 50.0; [6,9] probe 7 \u2192 33.33."),
    ("placement_error_from_true_position", "float", "slider_value \u2212 true_position_pct."),
    ("was_legacy_slider_pair", "bool", "True if one of the original 6 SLIDER_PAIRS."),
    ("", "", ""),
    ("middle_item_index", "int", "Legacy alias for placement_probe_index."),
    ("middle_item_img", "str", "Legacy alias for placement_probe_img."),
    ("middle_item_is_boundary", "int/null", "1=BOUNDARY_MIDDLE, 0=NONBOUNDARY_MIDDLE, null otherwise."),
    ("placement_true_midpoint_pct", "float", "Legacy alias for placement_true_position_pct. Now interpreted as true serial position, not literal midpoint."),
    ("placement_error_from_true_midpoint", "float", "Legacy alias for placement_error_from_true_position."),
]
dict_df = pd.DataFrame(dict_rows, columns=["field","type","description"])
display(dict_df[dict_df.field!=""])

html = styled_table_css("datadict") + '<div id="datadict">'
html += "<table><caption>Expected memory-task CSV data dictionary</caption>"
html += "<tr><th>field</th><th>type</th><th>description</th></tr>"
for _, row in dict_df.iterrows():
    if row.field == "": html += '<tr><td colspan="3" style="background:#e8e8e8;font-weight:700">Legacy compatibility fields</td></tr>'
    else: html += f"<tr><td><code>{row.field}</code></td><td>{row.type}</td><td>{row.description}</td></tr>"
html += "</table></div>"
save_html(html, "expected_memory_csv_dictionary.html")
print("\u2713 Data dictionary exported.")

,field,type,description
0,placement_probe_index,int,1-indexed within-block serial position of the ...
1,placement_probe_img,str,Stimulus path (PNG) or emoji string of the sel...
2,placement_probe_position,int,Ordinal among candidates (1 = earliest).
3,placement_candidate_indices,str,Comma-separated 1-indexed positions of all can...
4,placement_num_candidate_items,int,Number of candidates (1 or 2).
5,placement_trial_type,str,Detailed label: legacy pair status + distance ...
6,placement_true_position_pct,float,True serial position on slider (0–100). E.g. [...
7,placement_error_from_true_position,float,slider_value − true_position_pct.
8,was_legacy_slider_pair,bool,True if one of the original 6 SLIDER_PAIRS.
10,middle_item_index,int,Legacy alias for placement_probe_index.


  ✓ Exported design_checks/expected_memory_csv_dictionary.html
✓ Data dictionary exported.


## Consistency Checklist

In [15]:
checks = [
    ("A. Source files present and hashed", True),
    ("A. stimuli_version is PNG-based", "png" in (stim_version or "").lower()),
    ("B. 200 unique PNG main stimuli", len(set(STIMULUS_IMAGE_POOL))==200),
    ("B. 3 practice emoji (non-PNG)", len(PRACTICE_EMOJI)==3),
    ("C. 4-block design matches spec", True),
    ("D. 4\u00d750 trajectories, finite, bounded", True),
    ("E. Change points documented", True),
    ("F. 14 predefined pairs (6 d1, 8 d2)", len(PREDEFINED_PAIRS)==14),
    ("G. All-pairs slider: 14/block, 6+4+4 balance", True),
    ("H. Pair-order: no dups, broad coverage", True),
    ("I. Boundary classification exported", True),
    ("J. Image assignment + stimulus ribbons", True),
    ("K. 10 drop fragments, feedback mapping", len(drop_dist)==10),
    ("L. Data dictionary exported", True),
]

for label, passed in checks:
    print(f"  {'✅' if passed else '❌'}  {label}")

exports = sorted(DESIGN_DIR.glob("*"))
print(f"\n── All design checks complete. {len(exports)} files in design_checks/: ──")
for e in exports:
    print(f"   {e.name}  ({e.stat().st_size/1024:.1f} KB)")

  ✅  A. Source files present and hashed
  ✅  A. stimuli_version is PNG-based
  ✅  B. 200 unique PNG main stimuli
  ✅  B. 3 practice emoji (non-PNG)
  ✅  C. 4-block design matches spec
  ✅  D. 4×50 trajectories, finite, bounded
  ✅  E. Change points documented
  ✅  F. 14 predefined pairs (6 d1, 8 d2)
  ✅  G. All-pairs slider: 14/block, 6+4+4 balance
  ✅  H. Pair-order: no dups, broad coverage
  ✅  I. Boundary classification exported
  ✅  J. Image assignment + stimulus ribbons
  ✅  K. 10 drop fragments, feedback mapping
  ✅  L. Data dictionary exported

── All design checks complete. 40 files in design_checks/: ──
   .DS_Store  (8.0 KB)
   cb_priority_chain.html  (2.6 KB)
   change_point_map.html  (47.9 KB)
   change_point_map.png  (35.8 KB)
   consistency_report.html  (4.6 KB)
   design_table.html  (2.9 KB)
   drop_object_distribution.html  (58.3 KB)
   drop_object_distribution.png  (43.6 KB)
   drop_object_distribution_feedback_audit.html  (2.6 KB)
   emoji_ribbons.html  (30.1 KB)
   e